In [ ]:
import os
from pathlib import Path
import re

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from dna_features_viewer import GraphicFeature, GraphicRecord

cwd = os.getcwd()
if cwd.endswith('bacteriocins'):
    os.chdir('../..')
    cwd = os.getcwd()

In [6]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('data')
assert data_folder.is_dir()

lcna_folder = Path('data/amp_db/Lactococcin_A')
assert lcna_folder.is_dir()

In [3]:
gtdb_metadata = pd.read_csv(data_folder / 'amp_db' / 'gtdb_metadata.csv.gz', index_col='ncbi_accession')

In [4]:
def parse_gff(path : Path):
    gff_csv = pd.read_csv(
        path,
        sep='\t',
        comment='#',
        header=None,
        names=[
            'contig_id',
            'source',
            'type',
            'start',
            'end',
            'na1',
            'strand',
            'na2',
            'metadata',
        ],
    )

    def parse_protein_id(metadata : str):
        m = re.match(r'^.*protein_id=([^;]+).*$', metadata)
        if m is not None:
            return m[1]
        else:
            return None
        
    def parse_description(metadata : str):
        m = re.match(r'^.*product=([^;]+).*$', metadata)
        if m is not None:
            return m[1]
        else:
            return None

    gff_csv = gff_csv[gff_csv['type'] == 'CDS'].reset_index(drop=True)
    gff_csv['protein_id'] = gff_csv['metadata'].apply(parse_protein_id)
    gff_csv['description'] = gff_csv['metadata'].apply(parse_description)

    return gff_csv.drop_duplicates(['contig_id', 'protein_id']).reset_index(drop=True).set_index('protein_id')


In [7]:
m_bryantii_gff = parse_gff(lcna_folder / 'Methanobacterium bryantii' / 'GCF_002287175.1_ASM228717v1_genomic.gff')
m_bryantii_gff.head()

,contig_id,source,type,start,end,na1,strand,na2,metadata,description
protein_id,,,,,,,,,,
WP_069581966.1,NZ_LMVM01000001.1,Protein Homology,CDS,1228,2742,.,+,0,ID=cds-WP_069581966.1;Parent=gene-ASJ80_RS0000...,Mur ligase family protein
WP_069581964.1,NZ_LMVM01000001.1,Protein Homology,CDS,3087,3338,.,-,0,ID=cds-WP_069581964.1;Parent=gene-ASJ80_RS0001...,hypothetical protein
WP_176720166.1,NZ_LMVM01000001.1,Protein Homology,CDS,3733,6069,.,+,0,ID=cds-WP_176720166.1;Parent=gene-ASJ80_RS0001...,response regulator
WP_069581962.1,NZ_LMVM01000001.1,Protein Homology,CDS,6267,8162,.,-,0,ID=cds-WP_069581962.1;Parent=gene-ASJ80_RS0002...,helix-hairpin-helix domain-containing protein
WP_069581960.1,NZ_LMVM01000001.1,Protein Homology,CDS,8632,9441,.,+,0,ID=cds-WP_069581960.1;Parent=gene-ASJ80_RS0002...,sulfite exporter TauE/SafE family protein


In [9]:
m_bryantii_lcna = m_bryantii_gff.loc['WP_069585497.1']

n_upstream = n_downstream = 9_000
contig_id = m_bryantii_lcna['contig_id']
start = max(m_bryantii_lcna['start'] - n_upstream, 1)
end = m_bryantii_lcna['end'] + n_downstream

m_bryantii_lcna_context = m_bryantii_gff[
    (m_bryantii_gff['contig_id'] == contig_id) &
    (m_bryantii_gff['start'] >= start) &
    (m_bryantii_gff['start'] < end) &
    (m_bryantii_gff['end'] > start) &
    (m_bryantii_gff['end'] <= end)
].copy()
m_bryantii_lcna_context

,contig_id,source,type,start,end,na1,strand,na2,metadata,description
protein_id,,,,,,,,,,
WP_069585475.1,NZ_LMVM01000012.1,Protein Homology,CDS,158196,159488,.,-,0,ID=cds-WP_069585475.1;Parent=gene-ASJ80_RS0622...,TrpB-like pyridoxal phosphate-dependent enzyme
WP_083241061.1,NZ_LMVM01000012.1,Protein Homology,CDS,159759,162203,.,-,0,ID=cds-WP_083241061.1;Parent=gene-ASJ80_RS0622...,PAS domain S-box protein
WP_176720325.1,NZ_LMVM01000012.1,GeneMarkS-2+,CDS,162940,163116,.,-,0,ID=cds-WP_176720325.1;Parent=gene-ASJ80_RS1699...,hypothetical protein
WP_141705216.1,NZ_LMVM01000012.1,GeneMarkS-2+,CDS,163257,163748,.,-,0,ID=cds-WP_141705216.1;Parent=gene-ASJ80_RS1662...,hypothetical protein
WP_069585481.1,NZ_LMVM01000012.1,GeneMarkS-2+,CDS,164268,164756,.,+,0,ID=cds-WP_069585481.1;Parent=gene-ASJ80_RS0623...,PepSY domain-containing protein
WP_176720326.1,NZ_LMVM01000012.1,GeneMarkS-2+,CDS,164829,165218,.,-,0,ID=cds-WP_176720326.1;Parent=gene-ASJ80_RS0623...,hypothetical protein
WP_141705217.1,NZ_LMVM01000012.1,GeneMarkS-2+,CDS,165390,166022,.,-,0,ID=cds-WP_141705217.1;Parent=gene-ASJ80_RS0624...,hypothetical protein
WP_069585491.1,NZ_LMVM01000012.1,Protein Homology,CDS,166034,166318,.,-,0,ID=cds-WP_069585491.1;Parent=gene-ASJ80_RS0624...,hypothetical protein
WP_069585497.1,NZ_LMVM01000012.1,Protein Homology,CDS,166910,167692,.,-,0,ID=cds-WP_069585497.1;Parent=gene-ASJ80_RS0625...,cysteine peptidase family C39 domain-containin...
